# 06 — NBC News: scrape, NA check, author/date backfill, export

`scripts/scrape_nbc.py` collects **`author`** / **`datetime_posted`** per URL (live blogs use JSON-LD; others use **`cXenseParse:*`** metas); card-style live URLs and some article types still come out incomplete. **`scripts/backfill_nbc.py`** refetches rows with gaps (default **all**, not only **`/live-blog/`**; use **`--live-only`** for that scope) using Branch metas, JSON-LD with **shell–URL matching**, **`BlogPosting`**, **`cXenseParse`**, and **`article:published_time`**.

Flow: optional re-scrape → NA report → optional **backfill missing author/datetime** → **drop rows where title, subtitle, author, datetime_posted are all missing** → save `nbc_scraped_all_back.csv`.


## Part 1 — Refresh the NBC scrape (optional)

`scripts/scrape_nbc.py` reads `data/external/url_only_data.csv` and writes `data/raw/nbc_scraped_all.csv`. Live blogs use NBC page-level JSON-LD on that pass.

Set `RUN_BASE_SCRAPE = True` for a full re-scrape; otherwise reuse the CSV on disk.


In [19]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

# Notebook cwd is expected to be `notebooks/`
ROOT = Path.cwd().resolve().parent
BASE_CSV = ROOT / "data" / "raw" / "nbc_scraped_all.csv"

RUN_BASE_SCRAPE = False

if RUN_BASE_SCRAPE:
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "scrape_nbc.py")],
        cwd=str(ROOT),
        check=True,
    )
    print("Scrape finished:", BASE_CSV)
else:
    print("Skipping scrape (set RUN_BASE_SCRAPE = True to refresh).")
    print("Using existing:", BASE_CSV)


Skipping scrape (set RUN_BASE_SCRAPE = True to refresh).
Using existing: /Users/josephdattilo/Desktop/college/Senior/CIS 5190/cis-5190-news/data/raw/nbc_scraped_all.csv


## Part 2 — Missing values (NA)

Load the scrape CSV as-is and summarize where values are missing: counts per column, how many rows have any NA, and a short preview of URLs for each column with gaps.


In [20]:
nbc = pd.read_csv(BASE_CSV)

print(f"Rows: {len(nbc)}")
na_counts = nbc.isna().sum()
print("\nNA count per column:")
print(na_counts)

any_na = nbc.isna().any(axis=1)
print(f"\nRows with at least one NA: {int(any_na.sum())}")

cols = [c for c in nbc.columns if c != "label"]
for col in cols:
    miss = nbc[nbc[col].isna()]
    if miss.empty:
        continue
    print(f"\n--- `{col}` missing: {len(miss)} rows (sample) ---")
    show_cols = [c for c in ["url", "topic", "title", "subtitle", "author", "datetime_posted"] if c in nbc.columns]
    print(miss[show_cols].head(5).to_string(index=False))


Rows: 1805

NA count per column:
url                0
topic              0
title              4
subtitle           4
author             4
datetime_posted    5
label              0
dtype: int64

Rows with at least one NA: 5

--- `title` missing: 4 rows (sample) ---
                                                                                                     url   topic title subtitle author datetime_posted
                          https://www.nbcnews.com/select/shopping/select-pet-awards-under-25-ncna1306012  select   NaN      NaN    NaN             NaN
https://www.nbcnews.com/feature/nbc-out/lil-nas-x-dolly-parton-lena-waithe-appear-virtual-glaad-n1233424 feature   NaN      NaN    NaN             NaN
                         https://www.nbcnews.com/select/shopping/new-notable-sonos-jbl-fitbit-rcna156060  select   NaN      NaN    NaN             NaN
                           https://www.nbcnews.com/select/shopping/mr-coffee-mug-warmer-gift-ncna1285511  select   NaN      NaN    

## Part 3 — Backfill authors + date posted

**`backfill_nbc.py`** refetches any row with missing **`author`** and/or **`datetime_posted`** (default: all such rows, not only **`/live-blog/`**). **`--live-only`** narrows to **`/live-blog/`** like the old behavior.

Why gaps remain after the base scrape:

- **Live “card” URLs** (`…/rcrd…`, `?canonicalCard=`) extend the shell path, but JSON-LD still points at the **parent** live-blog URL — the scraper matched URLs **strictly**, so **`datePublished`** (and often authors) were skipped. Backfill uses **path-prefix–aware** JSON-LD matching (shared with **`scrape_nbc.py`** for new scrapes).
- **Non-live pages** (election results, specials, some blogs) often lack **`cXenseParse:*`** metas the scraper reads; JSON-LD **`NewsArticle`** / **`BlogPosting`** + **`cXense`** / **`article:published_time`** backfill those fields when present.

Priority order: Branch contributor metas (live) → JSON-LD author/date (loose URL match, `LiveBlogPosting` > `NewsArticle` > `BlogPosting`) → **`cXenseParse:author`** / **`cXenseParse:publishtime`** → **`article:published_time`**.

Writes **`data/raw/nbc_scraped_all.csv`** in place when you run the script below.


In [11]:
RUN_LIVE_BLOG_BACKFILL = False

live_blog_mask = nbc["url"].astype(str).str.contains("/live-blog/", case=False, na=False)
missing_author = nbc["author"].isna() | (nbc["author"].astype(str).str.strip() == "")
missing_dt = nbc["datetime_posted"].isna() | (
    nbc["datetime_posted"].astype(str).str.strip() == ""
)
lb_miss_author = live_blog_mask & missing_author
lb_miss_dt = live_blog_mask & missing_dt
need_any_field = missing_author | missing_dt
need_backfill = need_any_field  # matches script default (all gaps); use --live-only for live subset

n_live_blog = int(live_blog_mask.sum())
n_author_na_total = int(
    (nbc["author"].isna() | (nbc["author"].astype(str).str.strip() == "")).sum()
)
n_dt_na_total = int(
    (
        nbc["datetime_posted"].isna()
        | (nbc["datetime_posted"].astype(str).str.strip() == "")
    ).sum()
)

print(
    f"After loading the scrape ({len(nbc)} rows):\n"
    f"  • Missing author (any URL): {n_author_na_total}\n"
    f"  • Missing datetime_posted (any URL): {n_dt_na_total}\n"
    f"  • Live-blog URLs (/live-blog/): {n_live_blog}\n"
    f"  • Live-blog + missing author: {int(lb_miss_author.sum())}\n"
    f"  • Live-blog + missing datetime_posted: {int(lb_miss_dt.sum())}\n"
    f"  • Rows script will refetch (default, any URL with gaps): {int(need_backfill.sum())} "
    f"(live {int((live_blog_mask & need_any_field).sum())}, non-live {int((~live_blog_mask & need_any_field).sum())})\n"
    "→ Run backfill_nbc.py (add --live-only to limit to /live-blog/).\n"
)

if RUN_LIVE_BLOG_BACKFILL:
    subprocess.run(
        [sys.executable, str(ROOT / "scripts" / "backfill_nbc.py"), "--input", str(BASE_CSV)],
        cwd=str(ROOT),
        check=True,
    )
    nbc = pd.read_csv(BASE_CSV)
    print("Reloaded `nbc` after live-blog author + date-posted backfill.")
else:
    print("Skipping live-blog backfill (set RUN_LIVE_BLOG_BACKFILL = True).")


After loading the scrape (1805 rows):
  • Missing author (any URL): 13
  • Missing datetime_posted (any URL): 43
  • Live-blog URLs (/live-blog/): 83
  • Live-blog + missing author: 0
  • Live-blog + missing datetime_posted: 32
  • Live-blog rows needing author and/or date-posted backfill: 32
→ Run backfill_nbc.py: Branch meta → author; JSON-LD datePublished → datetime_posted.

Skipping live-blog backfill (set RUN_LIVE_BLOG_BACKFILL = True).


## Part 4 — Drop rows with no title, subtitle, author, or date posted

Remove rows where **`title`**, **`subtitle`**, **`author`**, and **`datetime_posted`** are **all** missing (`NaN` or blank after strip). Run this **after Part 3** so an optional CSV reload from **`backfill_nbc.py`** does not undo the drop.

Part 5 writes this in-memory **`nbc`** to **`nbc_scraped_all_back.csv`**; **`data/raw/nbc_scraped_all.csv`** stays whatever scrape/backfill last wrote.


In [22]:
def _field_missing(series: pd.Series) -> pd.Series:
    return series.isna() | (series.astype(str).str.strip() == "")


all_four_missing = (
    _field_missing(nbc["title"])
    | _field_missing(nbc["subtitle"])
    | _field_missing(nbc["author"])
    | _field_missing(nbc["datetime_posted"])
)
n_drop = int(all_four_missing.sum())
n_before = len(nbc)
nbc = nbc.loc[~all_four_missing].reset_index(drop=True)
print(
    f"Dropped {n_drop} row(s): {n_before} → {len(nbc)} "
    "(all of title, subtitle, author, datetime_posted missing).\n"
    "If any dropped URL should stay, inspect `nbc` before this cell or adjust the scrape."
)

print("\n--- After drop: NA counts (same as Part 2: pandas `isna`) ---")
print(f"Rows: {len(nbc)}")
na_after = nbc.isna().sum()
print("\nNA count per column:")
print(na_after)
print(f"\nRows with at least one NA: {int(nbc.isna().any(axis=1).sum())}")


Dropped 1 row(s): 1801 → 1800 (all of title, subtitle, author, datetime_posted missing).
If any dropped URL should stay, inspect `nbc` before this cell or adjust the scrape.

--- After drop: NA counts (same as Part 2: pandas `isna`) ---
Rows: 1800

NA count per column:
url                0
topic              0
title              0
subtitle           0
author             0
datetime_posted    0
label              0
dtype: int64

Rows with at least one NA: 0
